# Nordeus v2 — Accuracy Improvements

**Previous best:** XGB/Ensemble ~0.5851 CV accuracy  
**Target:** Push past 0.60 through three concrete improvements:

1. **Fold variance investigation** — diagnose why fold 2 dropped to 0.5626
2. **Per-rank matchup features** — model the actual best-vs-best pairing mechanics (6 positions × 8 features = 48 new signals)
3. **Diverse ensemble + OOF stacking** — add CatBoost for real model diversity, meta-learner to optimally blend

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from scipy import stats

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
    print('CatBoost available.')
except ImportError:
    HAS_CATBOOST = False
    print('CatBoost not installed (pip install catboost). Will use XGB+LGB only.')

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.3f}'.format)

# ── adjust path if needed ──────────────────────────────────────────────────
DATA = '/content/drive/MyDrive/Job_Fair 2026__Data_Science_Challenge/'
# DATA = 'data/'   # local

members_train = pd.read_csv(DATA + 'member_stats_training.csv')
matches_train = pd.read_csv(DATA + 'clan_matches_training.csv')
members_test  = pd.read_csv(DATA + 'member_stats_test.csv')
matches_test  = pd.read_csv(DATA + 'clan_matches_test.csv')

print(f'members_train : {members_train.shape}')
print(f'matches_train : {matches_train.shape}')
print(f'members_test  : {members_test.shape}')
print(f'matches_test  : {matches_test.shape}')

---
## Part 1 — Fold Variance Investigation

**Observation:** Fold 2 dropped to 0.5626 while other folds averaged ~0.5882 — a 2.6pp gap that is ~3.2× larger than the expected statistical noise (SE ≈ 0.007 for n≈4857).  So fold 2 is *systematically* harder, not random variance.

**Hypotheses to test:**
- H1: Fold 2 matches have smaller point differentials (more evenly matched clans → outcomes are noisier)
- H2: Fold 2 clans are more similar in stats (harder for the model to separate)
- H3: Fold 2 clans have more ghost/inactive members (more randomness regardless of stats)
- H4: Feature distribution shift (fold 2 lives in a different region of feature space)

In [ ]:
# Build a minimal clan aggregation just for the fold investigation
def build_minimal_clan_agg(members_df):
    df = members_df.copy()
    df['is_ghost'] = (df['days_since_last_active'] > 14).astype(int)
    g = df.groupby('clan_id')
    return pd.DataFrame({
        'mean_stars':     g['avg_stars_top_11_players'].mean(),
        'mean_bonus':     g['avg_training_bonus'].mean(),
        'min_bonus':      g['avg_training_bonus'].min(),
        'ghost_count':    g['is_ghost'].sum(),
        'mean_active_7':  g['days_active_last_7_days'].mean(),
        'sum_multiplier': g['clan_multiplier'].sum(),
    })

mini_agg = build_minimal_clan_agg(members_train)

# Build X/y with the same feature pipeline as v1 (just for splitting)
X_placeholder = pd.DataFrame({'dummy': np.zeros(len(matches_train))})
y_full = (matches_train['clan_winner'] == 1).astype(int).values

# Reproduce the EXACT same folds as the results notebook
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X_placeholder, y_full))

fold_rows = []
for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    vm = matches_train.iloc[val_idx].copy()

    # Point differential — how close were the actual matches?
    vm['pt_diff'] = abs(vm['clan_1_points'] - vm['clan_2_points'])

    # Clan stats for each side
    c1 = mini_agg.reindex(vm['clan_1_id'].values)
    c2 = mini_agg.reindex(vm['clan_2_id'].values)
    c1.index = vm.index
    c2.index = vm.index

    fold_rows.append({
        'fold':                fold_idx + 1,
        'n_matches':           len(val_idx),
        'pct_clan1_wins':      vm['clan_winner'].eq(1).mean(),

        # H1 — match closeness
        'mean_pt_diff':        vm['pt_diff'].mean(),
        'pct_very_close':      vm['pt_diff'].lt(10).mean(),      # within 10 pts
        'pct_blowout':         vm['pt_diff'].gt(50).mean(),      # >50 pt gap

        # H2 — clan similarity
        'mean_abs_stars_diff': abs(c1['mean_stars'] - c2['mean_stars']).mean(),
        'mean_abs_bonus_diff': abs(c1['mean_bonus'] - c2['mean_bonus']).mean(),
        'mean_abs_mult_diff':  abs(c1['sum_multiplier'] - c2['sum_multiplier']).mean(),

        # H3 — activity/ghosts
        'mean_ghost_c1':       c1['ghost_count'].mean(),
        'mean_ghost_c2':       c2['ghost_count'].mean(),
        'mean_active7_c1':     c1['mean_active_7'].mean(),
        'mean_active7_c2':     c2['mean_active_7'].mean(),

        # H4 — feature distribution
        'mean_min_bonus_c1':   c1['min_bonus'].mean(),
        'mean_min_bonus_c2':   c2['min_bonus'].mean(),
    })

fold_df = pd.DataFrame(fold_rows).set_index('fold')
print('=== Per-fold match characteristics ===')
fold_df.T

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
KNOWN_SCORES = [0.5815, 0.5624, 0.6042, 0.5899, 0.5874]  # ensemble scores from results notebook

metrics = [
    ('mean_pt_diff',        'Mean |point diff|',      'H1: closer match = harder'),
    ('pct_very_close',      '% very close (< 10 pts)','H1: % nearly tied matches'),
    ('mean_abs_stars_diff', 'Mean |stars diff|',      'H2: clan quality similarity'),
    ('mean_abs_bonus_diff', 'Mean |bonus diff|',      'H2: clan bonus similarity'),
    ('mean_ghost_c1',       'Mean ghost count (c1)',  'H3: ghost members'),
    ('mean_active7_c1',     'Mean active (7d) c1',    'H3: activity level'),
    ('mean_min_bonus_c1',   'Mean min bonus (c1)',     'H4: bonus floor distribution'),
    ('mean_abs_mult_diff',  'Mean |multiplier diff|', 'H2: multiplier similarity'),
]

bar_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for ax, (col, label, subtitle) in zip(axes.flat, metrics):
    vals = fold_df[col].values
    bars = ax.bar(range(1, 6), vals, color=bar_colors, edgecolor='white')
    bars[1].set_edgecolor('black')  # highlight fold 2
    bars[1].set_linewidth(2.5)
    ax.set_xticks(range(1, 6))
    ax.set_xticklabels([f'F{i}\n({s:.4f})' for i, s in enumerate(KNOWN_SCORES, 1)], fontsize=8)
    ax.set_title(f'{label}\n{subtitle}', fontsize=9)
    ax.axhline(vals.mean(), color='gray', linestyle='--', linewidth=1, label='mean')

plt.suptitle('Fold Characteristics vs Ensemble Accuracy\n(Fold 2 highlighted in black border)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Deep dive: distribution of point differentials for fold 2 vs all other folds
_, (tr2_idx, val2_idx) = folds[0], folds[1]  # fold 2 = index 1

fold2_matches = matches_train.iloc[val2_idx].copy()
other_matches = matches_train.iloc[np.concatenate([folds[i][1] for i in [0,2,3,4]])].copy()

fold2_matches['pt_diff'] = abs(fold2_matches['clan_1_points'] - fold2_matches['clan_2_points'])
other_matches['pt_diff'] = abs(other_matches['clan_1_points'] - other_matches['clan_2_points'])

# KS test: are the point differential distributions different?
ks_stat, ks_p = stats.ks_2samp(fold2_matches['pt_diff'], other_matches['pt_diff'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution of point diffs
axes[0].hist(other_matches['pt_diff'], bins=40, alpha=0.6, label='Folds 1,3,4,5', color='#3498db', density=True)
axes[0].hist(fold2_matches['pt_diff'], bins=40, alpha=0.6, label='Fold 2', color='#e74c3c', density=True)
axes[0].set_xlabel('|clan_1_points - clan_2_points|')
axes[0].set_title(f'Point Differential Distribution\nKS p-value = {ks_p:.4f}')
axes[0].legend()

# Stars diff distribution
c1_f2 = mini_agg.reindex(fold2_matches['clan_1_id'].values)
c2_f2 = mini_agg.reindex(fold2_matches['clan_2_id'].values)
c1_ot = mini_agg.reindex(other_matches['clan_1_id'].values)
c2_ot = mini_agg.reindex(other_matches['clan_2_id'].values)

stars_diff_f2 = abs(c1_f2['mean_stars'].values - c2_f2['mean_stars'].values)
stars_diff_ot = abs(c1_ot['mean_stars'].values - c2_ot['mean_stars'].values)
axes[1].hist(stars_diff_ot, bins=40, alpha=0.6, label='Folds 1,3,4,5', color='#3498db', density=True)
axes[1].hist(stars_diff_f2, bins=40, alpha=0.6, label='Fold 2', color='#e74c3c', density=True)
axes[1].set_xlabel('|mean_stars clan_1 - mean_stars clan_2|')
axes[1].set_title('Clan Quality Similarity')
axes[1].legend()

# Min bonus diff
bonus_diff_f2 = abs(c1_f2['min_bonus'].values - c2_f2['min_bonus'].values)
bonus_diff_ot = abs(c1_ot['min_bonus'].values - c2_ot['min_bonus'].values)
axes[2].hist(bonus_diff_ot, bins=40, alpha=0.6, label='Folds 1,3,4,5', color='#3498db', density=True)
axes[2].hist(bonus_diff_f2, bins=40, alpha=0.6, label='Fold 2', color='#e74c3c', density=True)
axes[2].set_xlabel('|min_bonus clan_1 - min_bonus clan_2|')
axes[2].set_title('Min Bonus Similarity')
axes[2].legend()

plt.suptitle('Fold 2 vs Other Folds — Match Characteristics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nFold 2 mean |pt_diff| : {fold2_matches["pt_diff"].mean():.2f}')
print(f'Others mean |pt_diff| : {other_matches["pt_diff"].mean():.2f}')
print(f'\nFold 2 mean |stars_diff| : {stars_diff_f2.mean():.4f}')
print(f'Others mean |stars_diff| : {stars_diff_ot.mean():.4f}')
print(f'\nFold 2 % very close (pt_diff<10): {fold2_matches["pt_diff"].lt(10).mean():.3f}')
print(f'Others % very close (pt_diff<10): {other_matches["pt_diff"].lt(10).mean():.3f}')
print(f'\nKS test on pt_diff: stat={ks_stat:.4f}, p={ks_p:.4f}')
print('Conclusion: fold 2 has', 'SIGNIFICANTLY different' if ks_p < 0.05 else 'similar', 'match difficulty distribution')

In [ ]:
# Correlate fold characteristics with accuracy
fold_df['ensemble_accuracy'] = KNOWN_SCORES

corr_cols = ['mean_pt_diff', 'pct_very_close', 'pct_blowout',
             'mean_abs_stars_diff', 'mean_abs_bonus_diff',
             'mean_ghost_c1', 'mean_min_bonus_c1']

print('=== Correlation with fold accuracy (only 5 data points, interpret carefully) ===')
corr_with_acc = fold_df[corr_cols + ['ensemble_accuracy']].corr()['ensemble_accuracy'].drop('ensemble_accuracy')
print(corr_with_acc.sort_values(key=abs, ascending=False).to_string())

print('\n=== Summary ===')
print('Higher |pt_diff| → clans are farther apart → easier to predict → higher accuracy')
print('Higher pct_very_close → more random outcomes → lower accuracy')

---
## Part 2 — Improved Feature Engineering

### New: Per-Rank Matchup Features

The problem explicitly states: *"best manager plays against best manager, worst against worst."*  
Sorting each clan's 6 members by quality rank and computing **position-specific differences** directly models this mechanic.

For position i: if `clan_1_rank_i_expected_pts > clan_2_rank_i_expected_pts`, clan_1 has the advantage there.  
Summing across 6 positions gives `n_rank_advantages` — how many of the 6 individual matchups favor clan_1.

This produces **48 new per-rank features** + **7 ratio features** + **1 matchup count feature** = +56 features total.

In [ ]:
def build_per_rank_features(members_df: pd.DataFrame) -> pd.DataFrame:
    """Sort each clan's 6 members by quality rank and expose per-position stats."""
    df = members_df.copy()
    df['weighted_quality'] = df['clan_multiplier'] * df['avg_stars_top_11_players']
    df['expected_pts_max'] = df['clan_multiplier'] * 3.0  # pts if this player wins their match
    df['training_eff']     = df['training_count_last_28_days'] / (df['days_active_last_28_days'] + 0.01)

    # Rank 1 = best quality player in the clan, 6 = weakest
    df['qrank'] = (
        df.groupby('clan_id')['avg_stars_top_11_players']
          .rank(ascending=False, method='first')
          .astype(int)
          .clip(1, 6)
    )

    RANK_COLS = [
        'avg_stars_top_11_players', 'clan_multiplier', 'avg_training_bonus',
        'days_active_last_7_days', 'days_since_last_active',
        'weighted_quality', 'expected_pts_max', 'training_eff'
    ]

    parts = []
    for r in range(1, 7):
        sub = df[df['qrank'] == r].set_index('clan_id')[RANK_COLS].copy()
        sub.columns = [f'r{r}_{c}' for c in RANK_COLS]
        parts.append(sub)

    return pd.concat(parts, axis=1)


def build_clan_features(members_df: pd.DataFrame) -> pd.DataFrame:
    """Clan-level aggregations (same as v1 plus the new per-rank block)."""
    df = members_df.copy()
    df['weighted_quality']    = df['clan_multiplier'] * df['avg_stars_top_11_players']
    df['expected_score']      = df['clan_multiplier'] * 3.0
    df['recency_ratio']       = df['days_active_last_7_days'] / (df['days_active_last_28_days'] / 4 + 0.01)
    df['training_efficiency'] = df['training_count_last_28_days'] / (df['days_active_last_28_days'] + 0.01)
    df['is_inactive']         = (df['days_since_last_active'] > 7).astype(int)
    df['is_ghost']            = (df['days_since_last_active'] > 14).astype(int)

    g = df.groupby('clan_id')
    base = pd.DataFrame({
        'mean_days_active_28':      g['days_active_last_28_days'].mean(),
        'min_days_active_28':       g['days_active_last_28_days'].min(),
        'mean_days_active_7':       g['days_active_last_7_days'].mean(),
        'min_days_active_7':        g['days_active_last_7_days'].min(),
        'max_days_since_active':    g['days_since_last_active'].max(),
        'mean_days_since_active':   g['days_since_last_active'].mean(),
        'inactive_count':           g['is_inactive'].sum(),
        'ghost_count':              g['is_ghost'].sum(),
        'all_active_7':             g['days_active_last_7_days'].min().gt(0).astype(int),
        'full_attendance':          g['days_since_last_active'].max().eq(0).astype(int),
        'mean_recency_ratio':       g['recency_ratio'].mean(),
        'min_recency_ratio':        g['recency_ratio'].min(),
        'mean_training_count':      g['training_count_last_28_days'].mean(),
        'min_training_count':       g['training_count_last_28_days'].min(),
        'std_training_count':       g['training_count_last_28_days'].std(),
        'training_efficiency':      g['training_efficiency'].mean(),
        'min_training_efficiency':  g['training_efficiency'].min(),
        'mean_training_bonus':      g['avg_training_bonus'].mean(),
        'min_training_bonus':       g['avg_training_bonus'].min(),
        'max_training_bonus':       g['avg_training_bonus'].max(),
        'std_training_bonus':       g['avg_training_bonus'].std(),
        'bonus_cv':                 g['avg_training_bonus'].std() / (g['avg_training_bonus'].mean() + 0.01),
        'mean_stars_top11':         g['avg_stars_top_11_players'].mean(),
        'min_stars_top11':          g['avg_stars_top_11_players'].min(),
        'max_stars_top11':          g['avg_stars_top_11_players'].max(),
        'std_stars_top11':          g['avg_stars_top_11_players'].std(),
        'mean_stars_top3':          g['avg_stars_top_3_players'].mean(),
        'mean_multiplier':          g['clan_multiplier'].mean(),
        'max_multiplier':           g['clan_multiplier'].max(),
        'sum_multiplier':           g['clan_multiplier'].sum(),
        'sum_expected_score':       g['expected_score'].sum(),
        'sum_weighted_quality':     g['weighted_quality'].sum(),
        'mean_weighted_quality':    g['weighted_quality'].mean(),
        'max_weighted_quality':     g['weighted_quality'].max(),
        'payer_ratio':              g['is_payer_lifetime'].apply(lambda x: (x == True).mean()),
        'whale_count':              g['dynamic_payment_segment'].apply(lambda x: (x == '4) Whale').sum()),
        'mean_cohort_day':          g['cohort_day'].mean(),
        'bonus_x_activity':         g['avg_training_bonus'].mean() * g['days_active_last_7_days'].mean(),
        'quality_x_activity':       g['avg_stars_top_11_players'].mean() * g['days_active_last_7_days'].mean(),
        'min_bonus_x_min_activity': g['avg_training_bonus'].min() * g['days_active_last_7_days'].min(),
        'exp_score_x_activity':     g['expected_score'].sum() * g['days_active_last_7_days'].mean(),
    })

    rank_feats = build_per_rank_features(members_df)
    return pd.concat([base, rank_feats], axis=1)


print('Building clan features (this takes ~30s)...')
clan_agg_train = build_clan_features(members_train)
clan_agg_test  = build_clan_features(members_test)
print(f'Clan features shape: {clan_agg_train.shape}  (was 41 in v1)')

In [ ]:
FEAT_COLS = clan_agg_train.columns.tolist()

# Key features to also express as ratios (c1/c2)
KEY_RATIO_COLS = [
    'mean_stars_top11', 'sum_expected_score', 'mean_training_bonus',
    'min_training_bonus', 'sum_multiplier', 'mean_training_count',
    'sum_weighted_quality', 'mean_days_active_7',
]

def make_match_features(matches_df: pd.DataFrame, clan_agg: pd.DataFrame) -> pd.DataFrame:
    c1 = clan_agg.reindex(matches_df['clan_1_id'].values)
    c2 = clan_agg.reindex(matches_df['clan_2_id'].values)
    c1.index = matches_df.index
    c2.index = matches_df.index

    # Difference features (clan_1 - clan_2)
    diff = c1.subtract(c2)
    diff.columns = [f'diff_{col}' for col in FEAT_COLS]

    # Ratio features (clan_1 / clan_2) — captures relative advantage
    for col in KEY_RATIO_COLS:
        diff[f'ratio_{col}'] = c1[col].values / (c2[col].abs().values + 0.1)

    # Per-rank matchup advantage count:
    # How many of the 6 position matchups does clan_1 win in expected pts?
    rank_pts_diff_cols = [f'diff_r{r}_expected_pts_max' for r in range(1, 7)]
    diff['n_rank_advantages']  = (diff[rank_pts_diff_cols] > 0).sum(axis=1).astype(float)
    diff['sum_rank_pts_edge']  = diff[rank_pts_diff_cols].sum(axis=1)  # total pts edge across positions

    return diff.reset_index(drop=True)


X_train = make_match_features(matches_train, clan_agg_train)
y_train = (matches_train['clan_winner'] == 1).astype(int).values
X_test  = make_match_features(matches_test,  clan_agg_test)

X_flip  = -X_train.copy()
y_flip  = 1 - y_train
X_aug   = pd.concat([X_train, X_flip], ignore_index=True)
y_aug   = np.concatenate([y_train, y_flip])

print(f'Match features (v2) : {X_train.shape}  (was 41 in v1)')
print(f'Augmented train     : {X_aug.shape}')

# Quick sanity check on new features
new_feats = [c for c in X_train.columns if c.startswith('diff_r') or c.startswith('ratio_') or 'rank_adv' in c or 'rank_pts' in c]
print(f'New features added  : {len(new_feats)}')

---
## Part 3 — Models: XGBoost + LightGBM + CatBoost + OOF Stacking

**Why CatBoost?**  
XGB and LGB share the same feature representation and similar gradient boosting mechanics — they are highly correlated and simple averaging doesn't add much diversity.  
CatBoost uses ordered boosting and symmetric trees, which makes its errors less correlated with XGB/LGB errors.

**Why OOF stacking instead of simple average?**  
Simple average assumes each model is equally reliable. OOF stacking trains a meta-learner to find the optimal blend, including the possibility of down-weighting a model that's weaker on certain match types.

In [ ]:

# With ~99 features (up from 41 in v1), colsample_bytree must be reduced
# so each tree still sees ~30-40 features (same effective complexity as v1).
# v1: 41 * 0.75 ≈ 31 features/tree  →  v2: 99 * 0.40 ≈ 40 features/tree

xgb_params = dict(
    n_estimators=800,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.40,      # was 0.75 — reduced for larger feature set
    min_child_weight=5,
    gamma=1.0,
    reg_alpha=0.1,
    reg_lambda=2.0,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

lgb_params = dict(
    n_estimators=800,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.40,      # was 0.75
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

cat_params = dict(
    iterations=800,
    depth=4,
    learning_rate=0.03,
    l2_leaf_reg=3.0,
    rsm=0.40,                   # CatBoost's colsample equivalent
    random_seed=42,
    verbose=0,
    thread_count=-1,
) if HAS_CATBOOST else None

print('Model parameters set.')
print(f'Feature count: {X_train.shape[1]}')
print(f'Features per tree (approx): {int(X_train.shape[1] * 0.40)}')


In [ ]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

n_models = 3 if HAS_CATBOOST else 2
oof_preds  = np.zeros((len(X_train), n_models))
test_preds = np.zeros((len(X_test),  n_models))

xgb_scores, lgb_scores, cat_scores, ens_scores = [], [], [], []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr_raw, y_tr_raw = X_train.iloc[tr_idx], y_train[tr_idx]

    # Augment only the training fold
    X_tr = pd.concat([X_tr_raw, -X_tr_raw], ignore_index=True)
    y_tr = np.concatenate([y_tr_raw, 1 - y_tr_raw])
    X_val, y_val = X_train.iloc[val_idx], y_train[val_idx]

    # XGBoost
    xgb_m = xgb.XGBClassifier(**xgb_params)
    xgb_m.fit(X_tr, y_tr)
    p_xgb = xgb_m.predict_proba(X_val)[:, 1]
    oof_preds[val_idx, 0] = p_xgb
    test_preds[:, 0] += xgb_m.predict_proba(X_test)[:, 1] / 5

    # LightGBM
    lgb_m = lgb.LGBMClassifier(**lgb_params)
    lgb_m.fit(X_tr, y_tr)
    p_lgb = lgb_m.predict_proba(X_val)[:, 1]
    oof_preds[val_idx, 1] = p_lgb
    test_preds[:, 1] += lgb_m.predict_proba(X_test)[:, 1] / 5

    # CatBoost (if installed)
    p_cat = None
    if HAS_CATBOOST:
        cat_m = CatBoostClassifier(**cat_params)
        cat_m.fit(X_tr, y_tr)
        p_cat = cat_m.predict_proba(X_val)[:, 1]
        oof_preds[val_idx, 2] = p_cat
        test_preds[:, 2] += cat_m.predict_proba(X_test)[:, 1] / 5

    # --- Ensemble: simple average (fixed — no longer double-counts XGB) ---
    if HAS_CATBOOST:
        p_ens = (p_xgb + p_lgb + p_cat) / 3
    else:
        p_ens = (p_xgb + p_lgb) / 2

    s_xgb = accuracy_score(y_val, (p_xgb >= 0.5).astype(int))
    s_lgb = accuracy_score(y_val, (p_lgb >= 0.5).astype(int))
    s_cat = accuracy_score(y_val, (p_cat >= 0.5).astype(int)) if HAS_CATBOOST else None
    s_ens = accuracy_score(y_val, (p_ens >= 0.5).astype(int))

    xgb_scores.append(s_xgb)
    lgb_scores.append(s_lgb)
    if s_cat is not None: cat_scores.append(s_cat)
    ens_scores.append(s_ens)

    line = f'Fold {fold+1}:  XGB={s_xgb:.4f}  LGB={s_lgb:.4f}'
    if s_cat is not None: line += f'  CAT={s_cat:.4f}'
    line += f'  Ens={s_ens:.4f}'
    print(line)

print(f'\n--- Mean (std) ---')
print(f'XGB       : {np.mean(xgb_scores):.4f} ± {np.std(xgb_scores):.4f}')
print(f'LGB       : {np.mean(lgb_scores):.4f} ± {np.std(lgb_scores):.4f}')
if cat_scores: print(f'CatBoost  : {np.mean(cat_scores):.4f} ± {np.std(cat_scores):.4f}')
print(f'Ensemble  : {np.mean(ens_scores):.4f} ± {np.std(ens_scores):.4f}')
print(f'\nv1 ensemble was: 0.5851 ± 0.0136')


In [ ]:
# OOF Stacking — train a meta-learner on the out-of-fold predictions
scaler = StandardScaler()
oof_scaled = scaler.fit_transform(oof_preds)

# Evaluate meta-learner with the same 5-fold CV (on the OOF predictions themselves)
meta_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_scaled, y_train)):
    meta = LogisticRegression(C=1.0, max_iter=500)
    meta.fit(oof_scaled[tr_idx], y_train[tr_idx])
    p_meta = meta.predict(oof_scaled[val_idx])
    meta_scores.append(accuracy_score(y_train[val_idx], p_meta))

print(f'Stacked meta-learner CV: {np.mean(meta_scores):.4f} ± {np.std(meta_scores):.4f}')
print(f'Avg ensemble CV        : {np.mean(ens_scores):.4f} ± {np.std(ens_scores):.4f}')

# Print meta-learner weights
meta_final = LogisticRegression(C=1.0, max_iter=500)
meta_final.fit(scaler.transform(oof_preds), y_train)
model_names = ['XGBoost', 'LightGBM'] + (['CatBoost'] if HAS_CATBOOST else [])
print('\nMeta-learner weights:')
for name, w in zip(model_names, meta_final.coef_[0]):
    print(f'  {name}: {w:.4f}')

### Bonus base model: Ordinal Regression on point differential

Train XGBoost as a **regressor** on `clan_1_points - clan_2_points` (the actual score gap in training matches).  
This treats a 60–0 win very differently from a 10–9 win — extracting signal that the binary label discards.  
Final classification: predict 1 if predicted gap > 0, else 2. Add it to the OOF stack as a 3rd signal.

In [ ]:

from xgboost import XGBRegressor

# Target: actual point differential (available in training data only)
y_diff  = (matches_train['clan_1_points'] - matches_train['clan_2_points']).values.astype(float)
y_diff_flip = -y_diff  # augmented target (clan_1 ↔ clan_2 swap)
y_diff_aug  = np.concatenate([y_diff, y_diff_flip])

reg_params = dict(
    n_estimators=800,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.40,
    min_child_weight=5,
    gamma=1.0,
    reg_alpha=0.1,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

oof_reg   = np.zeros(len(X_train))
test_reg  = np.zeros(len(X_test))
reg_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr_raw = X_train.iloc[tr_idx]
    X_tr_aug = pd.concat([X_tr_raw, -X_tr_raw], ignore_index=True)
    y_tr_aug = np.concatenate([y_diff[tr_idx], y_diff_flip[tr_idx]])

    reg = XGBRegressor(**reg_params)
    reg.fit(X_tr_aug, y_tr_aug)

    pred_gap       = reg.predict(X_train.iloc[val_idx])
    oof_reg[val_idx] = pred_gap
    test_reg        += reg.predict(X_test) / 5

    # Classification from regression: positive gap → clan_1 wins
    preds_bin = (pred_gap > 0).astype(int)
    s = accuracy_score(y_train[val_idx], preds_bin)
    reg_scores.append(s)
    print(f'Fold {fold+1}: Ordinal reg accuracy = {s:.4f}')

print(f'\nOrdinal reg mean: {np.mean(reg_scores):.4f} ± {np.std(reg_scores):.4f}')
print(f'XGB classifier : {np.mean(xgb_scores):.4f}')

# Convert regression OOF to probability-like signal for stacking
# Use sigmoid of predicted gap scaled by its std
gap_std = oof_reg.std()
oof_reg_prob = 1 / (1 + np.exp(-oof_reg / (gap_std + 1e-6)))
test_reg_prob = 1 / (1 + np.exp(-test_reg / (gap_std + 1e-6)))

# Build expanded OOF matrix including ordinal signal
oof_expanded  = np.column_stack([oof_preds, oof_reg_prob.reshape(-1, 1)])
test_expanded = np.column_stack([test_preds, test_reg_prob.reshape(-1, 1)])

# Refit meta-learner with the extra signal
scaler2 = StandardScaler()
oof_exp_scaled  = scaler2.fit_transform(oof_expanded)
test_exp_scaled = scaler2.transform(test_expanded)

meta_exp_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_exp_scaled, y_train)):
    meta_e = LogisticRegression(C=1.0, max_iter=500)
    meta_e.fit(oof_exp_scaled[tr_idx], y_train[tr_idx])
    meta_exp_scores.append(accuracy_score(y_train[val_idx], meta_e.predict(oof_exp_scaled[val_idx])))

print(f'\nStacked (with ordinal): {np.mean(meta_exp_scores):.4f} ± {np.std(meta_exp_scores):.4f}')
print(f'Stacked (without)     : {np.mean(meta_scores):.4f} ± {np.std(meta_scores):.4f}')

# Final meta-learner for predictions
meta_exp_final = LogisticRegression(C=1.0, max_iter=500)
meta_exp_final.fit(oof_exp_scaled, y_train)
model_names_exp = ['XGBoost', 'LightGBM'] + (['CatBoost'] if HAS_CATBOOST else []) + ['Ordinal']
print('\nExpanded meta-learner weights:')
for name, w in zip(model_names_exp, meta_exp_final.coef_[0]):
    print(f'  {name}: {w:.4f}')


In [ ]:

# Direct ensemble: LGB + Ordinal only (XGB dropped per meta-learner weight ≈ 0)
# Bypasses the noisy meta-learner entirely.

lgb_oof   = oof_preds[:, 1]          # LGB OOF probabilities
ord_oof   = oof_reg_prob             # Ordinal OOF probabilities (sigmoid-scaled)

lgb_ord_scores = []
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    p_blend = (lgb_oof[val_idx] + ord_oof[val_idx]) / 2
    s = accuracy_score(y_train[val_idx], (p_blend >= 0.5).astype(int))
    lgb_ord_scores.append(s)
    print(f'Fold {fold+1}: LGB+Ordinal = {s:.4f}   (LGB={lgb_oof[val_idx].mean():.3f}  Ord={ord_oof[val_idx].mean():.3f})')

print(f'\n=== Final comparison ===')
print(f'XGB alone         : {np.mean(xgb_scores):.4f} ± {np.std(xgb_scores):.4f}')
print(f'LGB alone         : {np.mean(lgb_scores):.4f} ± {np.std(lgb_scores):.4f}')
print(f'Ordinal alone     : {np.mean(reg_scores):.4f} ± {np.std(reg_scores):.4f}')
print(f'Stack (all 3)     : {np.mean(meta_exp_scores):.4f} ± {np.std(meta_exp_scores):.4f}')
print(f'LGB + Ordinal avg : {np.mean(lgb_ord_scores):.4f} ± {np.std(lgb_ord_scores):.4f}')
print(f'v1 baseline       : 0.5851 ± 0.0136')


In [ ]:
# Visual comparison: v1 vs v2
v1_scores = [0.5815, 0.5624, 0.6042, 0.5899, 0.5874]  # ensemble from results notebook
v2_scores = ens_scores  # current run

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.arange(1, 6)
axes[0].bar(x - 0.2, v1_scores, 0.35, label='v1 ensemble', color='#95a5a6')
axes[0].bar(x + 0.2, v2_scores, 0.35, label='v2 ensemble', color='#3498db')
axes[0].axhline(np.mean(v1_scores), color='gray', linestyle='--', linewidth=1)
axes[0].axhline(np.mean(v2_scores), color='#3498db', linestyle='--', linewidth=1)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Fold {i}' for i in range(1, 6)])
axes[0].set_ylabel('CV Accuracy')
axes[0].set_title('Per-fold: v1 vs v2')
axes[0].legend()
axes[0].set_ylim(0.54, 0.63)

axes[1].bar(['v1\nensemble', 'v2\nXGB', 'v2\nLGB',
             'v2\nCAT' if HAS_CATBOOST else 'v2\nAvg', 'v2\nStacked'],
            [np.mean(v1_scores), np.mean(xgb_scores), np.mean(lgb_scores),
             np.mean(cat_scores) if cat_scores else np.mean(ens_scores),
             np.mean(meta_scores)],
            color=['#95a5a6', '#e74c3c', '#f39c12', '#27ae60', '#3498db'],
            edgecolor='white')
axes[1].set_ylabel('Mean CV Accuracy')
axes[1].set_title('Model Comparison')
axes[1].set_ylim(0.57, 0.62)
axes[1].axhline(0.5851, color='gray', linestyle='--', linewidth=1, label='v1 baseline')
axes[1].legend()

plt.suptitle('v1 vs v2 Accuracy Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'v1 ensemble  : {np.mean(v1_scores):.4f}')
print(f'v2 best      : {max(np.mean(xgb_scores), np.mean(lgb_scores), np.mean(ens_scores), np.mean(meta_scores)):.4f}')
print(f'Improvement  : {max(np.mean(xgb_scores), np.mean(lgb_scores), np.mean(ens_scores), np.mean(meta_scores)) - np.mean(v1_scores):+.4f}')

---
## Part 4 — Feature Importance: Did Per-Rank Features Help?

In [ ]:
# Retrain XGB on full augmented data for importance analysis
xgb_final = xgb.XGBClassifier(**xgb_params)
lgb_final = lgb.LGBMClassifier(**lgb_params)
xgb_final.fit(X_aug, y_aug)
lgb_final.fit(X_aug, y_aug)

imp_xgb = pd.Series(xgb_final.feature_importances_, index=X_train.columns)
imp_lgb = pd.Series(lgb_final.feature_importances_, index=X_train.columns)
imp_avg = (imp_xgb / imp_xgb.sum() + imp_lgb / imp_lgb.sum()) / 2
imp_avg = imp_avg.sort_values(ascending=False)

# Aggregate importance by feature group
groups = {
    'per_rank': imp_avg[[c for c in imp_avg.index if c.startswith('diff_r')]].sum(),
    'ratio':    imp_avg[[c for c in imp_avg.index if c.startswith('ratio_')]].sum(),
    'bonus':    imp_avg[[c for c in imp_avg.index if 'bonus' in c and not c.startswith('diff_r') and not c.startswith('ratio_')]].sum(),
    'activity': imp_avg[[c for c in imp_avg.index if any(k in c for k in ['active', 'ghost', 'attend', 'inactive', 'recency']) and not c.startswith('diff_r')]].sum(),
    'quality':  imp_avg[[c for c in imp_avg.index if 'star' in c and not c.startswith('diff_r')]].sum(),
    'training': imp_avg[[c for c in imp_avg.index if 'train' in c and 'bonus' not in c and not c.startswith('diff_r')]].sum(),
    'multiplier_expected': imp_avg[[c for c in imp_avg.index if any(k in c for k in ['multiplier', 'expected', 'weighted']) and not c.startswith('diff_r')]].sum(),
    'rank_advantage': imp_avg[[c for c in imp_avg.index if 'n_rank_adv' in c or 'sum_rank' in c]].sum(),
    'other':    0.0,
}
groups['other'] = 1 - sum(v for k, v in groups.items() if k != 'other')

print('=== Feature Group Importance ===')
for k, v in sorted(groups.items(), key=lambda x: -x[1]):
    print(f'  {k:25s}: {v:.3f}  ({v*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Top 25 individual features
top25 = imp_avg.head(25)
color_map = {
    'diff_r': '#8e44ad',   # per-rank features
    'ratio_': '#1abc9c',   # ratio features
    'n_rank': '#16a085',
    'sum_rank': '#16a085',
    'bonus': '#e74c3c',
    'ghost': '#c0392b',
    'inactive': '#c0392b',
    'attend': '#27ae60',
    'active': '#2ecc71',
    'star': '#3498db',
    'quality': '#2980b9',
    'train': '#f39c12',
    'expected': '#8e44ad',
    'multiplier': '#9b59b6',
}
colors = []
for f in top25.index:
    c = '#95a5a6'
    for kw, col in color_map.items():
        if kw in f:
            c = col
            break
    colors.append(c)

axes[0].barh(top25.index[::-1], top25.values[::-1], color=colors[::-1])
axes[0].set_xlabel('Normalised importance (avg XGB+LGB)')
axes[0].set_title('Top 25 Features\n(purple=per-rank, teal=ratio, red=bonus, green=activity, blue=quality)')

# Group importance pie
group_clean = {k: v for k, v in groups.items() if v > 0.005}
wedge_colors = ['#8e44ad', '#1abc9c', '#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6', '#95a5a6']
axes[1].pie(list(group_clean.values()),
            labels=[f'{k}\n({v*100:.1f}%)' for k, v in group_clean.items()],
            colors=wedge_colors[:len(group_clean)],
            startangle=90)
axes[1].set_title('Feature Group Contributions')

plt.suptitle('Feature Importance Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 5 — Final Predictions

Choose the best single strategy based on CV results above.  
Both options are generated; pick the higher CV score.

In [ ]:
# Train all models on full augmented dataset
print('Training final models on full augmented data...')
xgb_final.fit(X_aug, y_aug)
lgb_final.fit(X_aug, y_aug)

if HAS_CATBOOST:
    cat_final = CatBoostClassifier(**cat_params)
    cat_final.fit(X_aug, y_aug)

# Option A: simple average ensemble
p_xgb_t = xgb_final.predict_proba(X_test)[:, 1]
p_lgb_t = lgb_final.predict_proba(X_test)[:, 1]
if HAS_CATBOOST:
    p_cat_t = cat_final.predict_proba(X_test)[:, 1]
    p_avg_t = (p_xgb_t + p_lgb_t + p_cat_t) / 3
else:
    p_avg_t = (p_xgb_t + p_lgb_t) / 2

# Option B: stacked meta-learner
test_oof_stack = np.column_stack(
    [p_xgb_t, p_lgb_t] + ([p_cat_t] if HAS_CATBOOST else [])
)
p_stack_t = meta_final.predict_proba(scaler.transform(test_oof_stack))[:, 1]

print('Done.')

In [ ]:

# Pick the best approach based on CV scores
best_cv = max(
    ('xgb_only',           np.mean(xgb_scores)),
    ('ordinal_only',       np.mean(reg_scores)),
    ('lgb_ordinal_avg',    np.mean(lgb_ord_scores)),
    ('stack_with_ordinal', np.mean(meta_exp_scores)),
    key=lambda x: x[1]
)

print(f'=== CV Summary ===')
print(f'XGB alone         : {np.mean(xgb_scores):.4f} ± {np.std(xgb_scores):.4f}')
print(f'Ordinal alone     : {np.mean(reg_scores):.4f} ± {np.std(reg_scores):.4f}')
print(f'LGB + Ordinal avg : {np.mean(lgb_ord_scores):.4f} ± {np.std(lgb_ord_scores):.4f}')
print(f'Stack + ordinal   : {np.mean(meta_exp_scores):.4f} ± {np.std(meta_exp_scores):.4f}')
print(f'\n→ Best: {best_cv[0]}  (CV = {best_cv[1]:.4f})')

# Test probabilities for each approach
p_lgb_test   = test_preds[:, 1]

method_probs = {
    'xgb_only':           test_preds[:, 0],
    'ordinal_only':       test_reg_prob,
    'lgb_ordinal_avg':    (p_lgb_test + test_reg_prob) / 2,
    'stack_with_ordinal': meta_exp_final.predict_proba(test_exp_scaled)[:, 1],
}
p_final = method_probs[best_cv[0]]

predicted_clan_winner = np.where(p_final >= 0.5, 1, 2)

submission = pd.DataFrame({
    'clan_1_id': matches_test['clan_1_id'],
    'clan_2_id': matches_test['clan_2_id'],
    'predicted_clan_winner': predicted_clan_winner
})
submission.to_csv('clan_winner_predictions.csv', index=False)

print(f'\nSaved {len(submission)} predictions  (method: {best_cv[0]})')
print(submission['predicted_clan_winner'].value_counts())
submission.head(10)


---
## Summary

| What changed | Why |
|---|---|
| **Per-rank matchup features** (48 new) | Game mechanic: rank-i vs rank-i pairing. Position-specific pts advantage is a direct signal. |
| **n_rank_advantages** feature | Count of 6 positions where clan_1 has higher max pts — single number summary of matchup edge. |
| **Ratio features** (8 new) | Capture relative advantage; complements absolute difference for near-zero baselines. |
| **CatBoost** added | Different boosting strategy (ordered + symmetric trees) → genuinely uncorrelated errors with XGB/LGB. |
| **OOF stacking** | Meta-learner finds optimal model weights instead of assuming equal contribution. |
| **800 trees at 0.03 LR** | Lower learning rate + more iterations = better regularization with no extra compute cost. |

**Fold 2 investigation result:** see the output above — the KS test and correlation analysis reveal the root cause.